# 5. Modelo aditivo + XGBoost con correcciones

**Objetivo:** superar al mejor modelo del episodio investigando una representación diferente de las variables.

La primera etapa es una logística regularizada con splines de ingreso y distancia. La segunda aprende una corrección en log-odds mediante `base_margin`. No se entrenan árboles sobre residuos con pérdida cuadrática: se conserva la pérdida logística.

Se comparan bloques de ingreso, movilidad, perfil de adopción y codificación de residuos por ingreso. Son hipótesis a validar, no mejoras demostradas.

**Antecedentes:** logit crudo 0,93810 CV; logit con derivadas 0,93855; XGBoost crudo 0,94186 CV / 0,94185 OOF; XGBoost con dos derivadas 0,94184 CV; **XGBoost tuneado con Optuna 0,94258 CV / 0,94257 OOF, 0,94242 en el leaderboard — ésa es la referencia a batir.**

> **Sobre qué baseline usan las comparaciones.** Las celdas de la sección 5 calculan los incrementos contra `previous_xgb`, el XGBoost **sin** tunear, porque se escribieron cuando `4_xgboost_optuna.ipynb` estaba a medio ejecutar. El modelo de Optuna sí entra en la tabla resumen y es el que corresponde mirar: contra él el delta del candidato es **+0,0026**, no el +0,0033 que sale contra el XGBoost crudo. Lo mismo vale para la mezcla 90/10 de la sección 6. La sección de resultados al final usa la referencia correcta.

**Uso:** ejecutar primero con `SMOKE=True`; luego cambiar a `False` y reiniciar el kernel para obtener OOF y submissions completos. El modo completo puede ser costoso en CPU: son seis variantes de XGBoost por fold. No se envía nada a Kaggle automáticamente.


In [1]:
from pathlib import Path
import os
import json
import hashlib
import time
import platform
import importlib.metadata as metadata

import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import SplineTransformer, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

SEED = 42
# La variable de entorno permite ejecutar una comprobación sin editar el notebook.
SMOKE = os.environ.get("EV_SMOKE", "0") == "1"  # cambiar a True para prueba rápida
N_FOLDS = 2 if SMOKE else 5
INNER_FOLDS = 2 if SMOKE else 3
N_TREES = 30 if SMOKE else 600
N_JOBS = 4
ALPHA = 100.0  # pseudoconteo para regularizar la codificación de residuos
TARGET = "Will_Buy_EV"
EXPERIMENTS = [
    "hybrid", "hybrid_income", "hybrid_mobility",
    "hybrid_profile", "hybrid_residual", "hybrid_all",
]
# Elección previa, no seleccionada automáticamente por el mayor OOF.
SUBMISSION_MODEL = "hybrid_all"

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/train.csv").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Abrir el notebook desde notebook/ o desde la raíz del episodio.")
OUT = ROOT / "models" / ("5_hybrid_smoke" if SMOKE else "5_hybrid")
OUT.mkdir(parents=True, exist_ok=True)
VERSIONS = {p: metadata.version(p) for p in
            ["numpy", "pandas", "scipy", "scikit-learn", "xgboost"]}
print("Modo:", "SMOKE: métricas NO comparables con CV completa" if SMOKE else "COMPLETO")
print("Versiones:", VERSIONS)
print("Salida:", OUT)


Modo: COMPLETO
Versiones: {'numpy': '2.0.2', 'pandas': '2.2.2', 'scipy': '1.14.1', 'scikit-learn': '1.7.2', 'xgboost': '2.1.3'}
Salida: c:\Users\HP\OneDrive\Escritorio\David Guzzi\Github\DGKaggle\Playground Series\Season 6\Episode 9 - Predicting Electric Vehicle Purchases\models\5_hybrid


## Datos y contrato de validación

Se conserva el orden del CSV para alinear las predicciones antiguas. Los folds exteriores completos replican `StratifiedKFold(5, shuffle=True, random_state=42)` de los notebooks anteriores.

Dentro de cada entrenamiento exterior se generan márgenes y codificaciones mediante folds internos. Cada fila de entrenamiento recibe variables aprendidas sin usar su etiqueta. El fold exterior jamás participa en ajuste de splines, vocabularios, frecuencias ni residuos.

Se fijan 600 árboles antes de evaluar. No hay early stopping sobre el fold exterior. Si se ajustan estos parámetros mirando los resultados, la comparación pasa a ser exploratoria y requiere confirmación independiente.

In [2]:
train = pd.read_csv(ROOT / "data/train.csv")
test = pd.read_csv(ROOT / "data/test.csv")
sample = pd.read_csv(ROOT / "data/sample_submission.csv")
assert train[TARGET].isin(["Yes", "No"]).all()
assert train["id"].is_unique and test["id"].is_unique
assert test["id"].equals(sample["id"])
original_rows = np.arange(len(train))

if SMOKE:
    original_rows, _ = train_test_split(
        original_rows, train_size=min(12000, len(train) - 2),
        stratify=train[TARGET], random_state=SEED)
    original_rows = np.sort(original_rows)
    train = train.iloc[original_rows].reset_index(drop=True)
    test = test.iloc[:min(1500, len(test))].copy().reset_index(drop=True)
    sample = sample.iloc[:len(test)].copy().reset_index(drop=True)

y = train[TARGET].eq("Yes").to_numpy(dtype=np.int8)
X = train.drop(columns=["id", TARGET])
XT = test.drop(columns="id")
assert set(X.columns) == set(XT.columns)
XT = XT[X.columns]
CAT = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()
NUM = [c for c in X if c not in CAT]
print(f"Train: {X.shape}; test: {XT.shape}; positivos: {y.mean():.4%}")
print("Repetición de ingresos (diagnóstico, sin usar target):")
for label, values in [("exacto", X["Annual_Income_USD"]),
                       ("redondeado a 100", X["Annual_Income_USD"].round(-2))]:
    counts = values.value_counts()
    print(label, "valores únicos:", len(counts),
          "| proporción de filas con valor repetido:", values.duplicated(False).mean())


Train: (668665, 13); test: (286571, 13); positivos: 17.4645%
Repetición de ingresos (diagnóstico, sin usar target):
exacto valores únicos: 13214 | proporción de filas con valor repetido: 0.9944142433056912
redondeado a 100 valores únicos: 1184 | proporción de filas con valor repetido: 0.9999446658640725


## 1. Componente aditivo flexible

El ingreso y la distancia tienen curvas cúbicas con seis nudos por cuantiles. Preocupación ambiental se representa como categoría, junto con las categóricas originales y el cruce carga doméstica × ciudad. Las otras numéricas entran linealmente. El escalado y la regularización se ajustan dentro de cada entrenamiento interno.

No se infiere la transformación del ingreso a partir de saltos entre deciles: esos intervalos pueden tener anchos distintos.

In [3]:
SMOOTH = ["Annual_Income_USD", "Daily_Commute_km"]
ADDITIVE_CAT = CAT + ["env_category", "home_city"]
LINEAR = [c for c in NUM if c not in SMOOTH + ["Environmental_Concern_Level"]]

def additive_frame(df):
    d = df.copy()
    d["env_category"] = d["Environmental_Concern_Level"].astype(str)
    d["home_city"] = (d["Home_Charging_Possible"].astype(str)
                      + "|" + d["City_Type"].astype(str))
    for c in ADDITIVE_CAT:
        d[c] = d[c].fillna("__MISSING__").astype(str)
    return d

def make_additive():
    pre = ColumnTransformer([
        ("smooth", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("spline", SplineTransformer(n_knots=6, degree=3,
                knots="quantile", include_bias=False, extrapolation="linear")),
        ]), SMOOTH),
        ("linear", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), LINEAR),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ADDITIVE_CAT),
    ], sparse_threshold=1.0)
    return Pipeline([
        ("pre", pre),
        ("scale", StandardScaler(with_mean=False)),
        ("model", LogisticRegression(C=0.1, max_iter=600,
                                      solver="lbfgs", random_state=SEED)),
    ])


## 2. Variables determinísticas: distintas representaciones del mismo dato

- **Ingreso:** tramo de USD 5.000, posición dentro del tramo, distancia al múltiplo de USD 1.000 más cercano y redondeo a centenas. Se conserva el ingreso crudo. No se supone que los últimos dígitos tengan señal.
- **Movilidad:** dependencia de carga pública, presión de distancia sobre estaciones, balance casa/trabajo, ausencia de estaciones y ansiedad combinada con falta de carga doméstica. Son proxies, no medidas físicas de capacidad de carga.
- **Perfil:** cruce de preocupación ambiental, subsidio y ansiedad; también carga doméstica × ciudad.

Un árbol podría reconstruir varios de estos cruces. Dárselos explícitos cambia su costo en profundidad y puede ayudar o perjudicar: las ablaciones lo medirán.

In [4]:
def deterministic_frame(df):
    d = df.copy()
    income = d["Annual_Income_USD"]
    d["income_band_5k"] = np.floor(income / 5000)
    d["income_position_5k"] = income / 5000 - d["income_band_5k"]
    d["income_distance_1k"] = np.abs(income / 1000 - np.round(income / 1000))
    d["income_round_100"] = np.round(income / 100) * 100

    no_home = d["Home_Charging_Possible"].eq("No").astype(float)
    h = d["Charging_Stations_Near_Home"]
    w = d["Charging_Stations_Near_Work"]
    commute = d["Daily_Commute_km"]
    d["public_commute"] = no_home * commute
    d["charging_min"] = np.minimum(h, w)
    d["charging_max"] = np.maximum(h, w)
    d["charging_imbalance"] = np.abs(h - w)
    d["no_stations_home"] = h.eq(0).astype(float)
    d["no_stations_work"] = w.eq(0).astype(float)
    d["public_friction"] = no_home * commute / (1 + h + w)
    d["anxiety_without_home"] = (
        d["Range_Anxiety_Level"].map({"Low": 0, "Medium": 1, "High": 2}) * no_home)
    d["adoption_profile"] = (
        d["Environmental_Concern_Level"].astype(str) + "|"
        + d["Subsidy_Available"].astype(str) + "|"
        + d["Range_Anxiety_Level"].astype(str))
    d["home_city"] = (
        d["Home_Charging_Possible"].astype(str) + "|" + d["City_Type"].astype(str))
    return d.replace([np.inf, -np.inf], np.nan)

INCOME_COLS = ["income_band_5k", "income_position_5k",
               "income_distance_1k", "income_round_100",
               "income_freq_exact", "income_freq_100"]
MOBILITY_COLS = ["public_commute", "charging_min", "charging_max",
                 "charging_imbalance", "no_stations_home", "no_stations_work",
                 "public_friction", "anxiety_without_home"]
PROFILE_COLS = ["adoption_profile", "home_city"]
RESIDUAL_COLS = ["income_residual_exact", "income_residual_100"]
ALL_NEW = INCOME_COLS + MOBILITY_COLS + PROFILE_COLS + RESIDUAL_COLS
BLOCKS = {
    "hybrid": [],
    "hybrid_income": INCOME_COLS,
    "hybrid_mobility": MOBILITY_COLS,
    "hybrid_profile": PROFILE_COLS,
    "hybrid_residual": RESIDUAL_COLS,
    "hybrid_all": ALL_NEW,
}


## 3. Correcciones locales de ingreso sin fuga del target

Para cada entrenamiento interno se ajusta el componente aditivo. Sobre **ese mismo entrenamiento interno** se obtienen residuos `y - p_aditivo` y se agregan por ingreso:

\[
\operatorname{correccion}(v)=\frac{\sum_{i:x_i=v}(y_i-p_i)}{n_v+100}.
\]

Se aplican los mapas al bloque interno excluido y al exterior/test. La predicción in-sample utilizada para estimar los residuos de referencia puede contraerlos; no es una medición de rendimiento. La fila que recibe la codificación nunca estuvo en ese ajuste.

Los valores desconocidos reciben frecuencia y corrección cero. Se usa ingreso exacto y redondeado a USD 100. Los mapas y márgenes de exterior/test se promedian entre modelos internos. Esto es bagging interno: el entrenamiento recibe una predicción excluida por fila, y exterior/test un promedio de modelos, sin reentrenamiento global adicional.

**No se usa el OOF antiguo como feature.** Solo se carga al final para comparar métricas.

In [5]:
LEARNED = ["income_freq_exact", "income_freq_100"] + RESIDUAL_COLS

def income_keys(df):
    return {
        "exact": df["Annual_Income_USD"],
        "100": np.round(df["Annual_Income_USD"] / 100) * 100,
    }

def fit_income_maps(df, residual):
    result = {}
    for name, key in income_keys(df).items():
        tab = pd.DataFrame({"key": key.to_numpy(), "residual": residual})
        agg = tab.groupby("key")["residual"].agg(["sum", "count"])
        result[name] = {
            "freq": agg["count"] / len(df),
            "residual": agg["sum"] / (agg["count"] + ALPHA),
        }
    return result

def apply_income_maps(df, maps):
    out = pd.DataFrame(index=df.index)
    for name, key in income_keys(df).items():
        out[f"income_freq_{name}"] = key.map(maps[name]["freq"]).fillna(0)
        out[f"income_residual_{name}"] = key.map(maps[name]["residual"]).fillna(0)
    return out[LEARNED]

def stage_one(x_train, y_train, x_valid, x_test, seed):
    # Índices locales para que la asignación OOF sea inequívoca.
    a = x_train.reset_index(drop=True)
    b = x_valid.reset_index(drop=True)
    c = x_test.reset_index(drop=True)
    aa, bb, cc = map(additive_frame, [a, b, c])
    n = len(a)
    margin_train = np.full(n, np.nan)
    margin_valid = np.zeros(len(b))
    margin_test = np.zeros(len(c))
    learned_train = pd.DataFrame(np.nan, index=a.index, columns=LEARNED)
    learned_valid = pd.DataFrame(0.0, index=b.index, columns=LEARNED)
    learned_test = pd.DataFrame(0.0, index=c.index, columns=LEARNED)
    coverage = np.zeros(n, dtype=int)
    inner = StratifiedKFold(INNER_FOLDS, shuffle=True, random_state=seed)

    for fit_idx, hold_idx in inner.split(a, y_train):
        assert not np.intersect1d(fit_idx, hold_idx).size
        model = make_additive()
        model.fit(aa.iloc[fit_idx], y_train[fit_idx])
        margin_train[hold_idx] = model.decision_function(aa.iloc[hold_idx])
        margin_valid += model.decision_function(bb) / INNER_FOLDS
        margin_test += model.decision_function(cc) / INNER_FOLDS

        fit_residual = y_train[fit_idx] - model.predict_proba(aa.iloc[fit_idx])[:, 1]
        maps = fit_income_maps(a.iloc[fit_idx], fit_residual)
        learned_train.iloc[hold_idx] = apply_income_maps(a.iloc[hold_idx], maps).to_numpy()
        learned_valid += apply_income_maps(b, maps) / INNER_FOLDS
        learned_test += apply_income_maps(c, maps) / INNER_FOLDS
        coverage[hold_idx] += 1

    assert np.all(coverage == 1)
    for obj in [margin_train, margin_valid, margin_test,
                learned_train, learned_valid, learned_test]:
        assert np.isfinite(np.asarray(obj)).all()
    return (margin_train, margin_valid, margin_test,
            learned_train, learned_valid, learned_test)

def tree_frames(a, b, c, la, lb, lc):
    frames = []
    for raw, learned in [(a, la), (b, lb), (c, lc)]:
        f = deterministic_frame(raw.reset_index(drop=True))
        f[LEARNED] = learned.to_numpy()
        frames.append(f)
    # Vocabulario calculado solo en entrenamiento exterior.
    for col in CAT + PROFILE_COLS:
        levels = sorted(frames[0][col].dropna().astype(str).unique())
        dtype = pd.CategoricalDtype(levels, ordered=False)
        for f in frames:
            f[col] = f[col].astype(dtype)
    return frames


## 4. Validación y ablaciones

Se comparte la primera etapa entre las variantes de un mismo fold para ahorrar trabajo. Todas usan los mismos márgenes, parámetros y particiones: cambia únicamente el bloque de variables.

El aditivo se evalúa también por separado. Los checkpoints por fold contienen índices y predicciones, pero esta versión vuelve a ejecutar todos los folds si se reinicia. No mezcla checkpoints de configuraciones distintas.

In [6]:
PARAMS = dict(
    objective="binary:logistic", eval_metric="auc",
    n_estimators=N_TREES, learning_rate=0.05, max_depth=4,
    min_child_weight=20, subsample=0.9, colsample_bytree=1.0,
    reg_lambda=10.0, reg_alpha=0.1,
    tree_method="hist", enable_categorical=True, max_bin=256,
    n_jobs=N_JOBS, random_state=SEED,
)
assert set(EXPERIMENTS).issubset(BLOCKS)
names = ["additive"] + EXPERIMENTS
oof = {name: np.full(len(X), np.nan, dtype=np.float32) for name in names}
test_pred = {name: np.zeros(len(XT), dtype=np.float64) for name in names}
fold_ids = np.full(len(X), -1, dtype=np.int8)
rows = []
outer = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)
started = time.perf_counter()

for fold, (tr, va) in enumerate(outer.split(X, y), 1):
    fold_start = time.perf_counter()
    fold_ids[va] = fold
    a, b = X.iloc[tr], X.iloc[va]
    print(f"Fold {fold}/{N_FOLDS}: primera etapa...", flush=True)
    mt, mv, ms, lt, lv, ls = stage_one(a, y[tr], b, XT, seed=SEED + fold)
    fa, fb, fc = tree_frames(a, b, XT, lt, lv, ls)
    fold_predictions = {"additive": (expit(mv), expit(ms))}

    for name in EXPERIMENTS:
        cols = list(X.columns) + BLOCKS[name]
        model = XGBClassifier(**PARAMS)
        model.fit(fa[cols], y[tr], base_margin=mt, verbose=False)
        pv = model.predict_proba(fb[cols], base_margin=mv)[:, 1]
        pt = model.predict_proba(fc[cols], base_margin=ms)[:, 1]
        fold_predictions[name] = (pv, pt)
        print(f"  {name:18s} AUC={roc_auc_score(y[va], pv):.6f}", flush=True)

    for name, (pv, pt) in fold_predictions.items():
        assert np.isfinite(pv).all() and np.isfinite(pt).all()
        assert ((pv >= 0) & (pv <= 1)).all()
        assert ((pt >= 0) & (pt <= 1)).all()
        oof[name][va] = pv
        test_pred[name] += pt / N_FOLDS
        rows.append({"fold": fold, "model": name,
                     "auc": roc_auc_score(y[va], pv)})
    checkpoint = {"valid_indices": va, "train_ids": train["id"].to_numpy()[va]}
    for name, (pv, pt) in fold_predictions.items():
        checkpoint[f"{name}_valid"] = pv
        checkpoint[f"{name}_test"] = pt
    np.savez_compressed(OUT / f"fold_{fold}.npz", **checkpoint)
    print(f"  Fold terminado en {(time.perf_counter()-fold_start)/60:.1f} min", flush=True)

assert (fold_ids > 0).all()
for name in names:
    assert np.isfinite(oof[name]).all()
print(f"Tiempo total: {(time.perf_counter()-started)/60:.1f} min")


Fold 1/5: primera etapa...
  hybrid             AUC=0.940805
  hybrid_income      AUC=0.942478
  hybrid_mobility    AUC=0.940749
  hybrid_profile     AUC=0.940787
  hybrid_residual    AUC=0.943616
  hybrid_all         AUC=0.944176
  Fold terminado en 6.0 min
Fold 2/5: primera etapa...
  hybrid             AUC=0.941651
  hybrid_income      AUC=0.943288
  hybrid_mobility    AUC=0.941629
  hybrid_profile     AUC=0.941585
  hybrid_residual    AUC=0.944406
  hybrid_all         AUC=0.944926
  Fold terminado en 5.2 min
Fold 3/5: primera etapa...
  hybrid             AUC=0.943070
  hybrid_income      AUC=0.944495
  hybrid_mobility    AUC=0.943038
  hybrid_profile     AUC=0.943069
  hybrid_residual    AUC=0.945404
  hybrid_all         AUC=0.946113
  Fold terminado en 5.9 min
Fold 4/5: primera etapa...
  hybrid             AUC=0.942476
  hybrid_income      AUC=0.943881
  hybrid_mobility    AUC=0.942453
  hybrid_profile     AUC=0.942430
  hybrid_residual    AUC=0.944882
  hybrid_all         AUC=0

## 5. Comparación pareada con los modelos guardados

Se distingue AUC OOF global de media de AUC por fold. La ganancia de cada bloque se compara contra `hybrid` — ése es el control interno correcto, porque comparte márgenes, hiperparámetros y particiones con las variantes, y lo único que cambia es el bloque de features.

**Atención al baseline externo.** La tabla "Incremento respecto del XGBoost anterior" usa `previous_xgb`, el XGBoost **sin** tunear (0,94185). El mejor modelo previo es `previous_optuna` (0,94257), que aparece en la tabla resumen pero no en esa comparación. Contra Optuna el delta del candidato es **+0,00259**, no +0,00331. Los deltas pareados contra Optuna, por fold, son 0,00279 / 0,00265 / 0,00255 / 0,00235 / 0,00256 — gana en 5 de 5, con un desvío de la diferencia de 0,00016.

Los archivos antiguos no incluyen IDs: se comprueba longitud, pero su alineación depende de que mantengan el orden original de `train.csv`, como establece el notebook 3. Los archivos nuevos sí guardan IDs y folds.

Los dos modelos previos usaron early stopping sobre el fold evaluado y, en el caso de Optuna, hiperparámetros distintos (`max_depth=11` contra el 4 de acá). La comparación es útil para iterar, aunque sus protocolos no son exactamente iguales. No se interpreta un t-test de cinco folds como confirmación formal: sus entrenamientos se solapan y se están comparando varias variantes.


In [7]:
fold_scores = pd.DataFrame(rows)
summary = pd.DataFrame([
    {"model": name, "auc_oof": roc_auc_score(y, oof[name]),
     "auc_fold_mean": fold_scores.loc[fold_scores.model.eq(name), "auc"].mean(),
     "auc_fold_std": fold_scores.loc[fold_scores.model.eq(name), "auc"].std(ddof=1)}
    for name in names
]).set_index("model")

if not SMOKE:
    for name, filename in [("previous_logit", "2_logit_oof.npy"),
                            ("previous_xgb", "3_xgboost_oof.npy"),
                            ("previous_optuna", "4_xgboost_optuna_oof.npy")]:
        path = ROOT / "models" / filename
        if not path.exists():
            print("No disponible:", filename)
            continue
        pred = np.load(path)
        if pred.shape != y.shape or not np.isfinite(pred).all():
            raise ValueError(f"OOF anterior inválido: {filename}")
        oof[name] = pred
        scores = [roc_auc_score(y[fold_ids == f], pred[fold_ids == f])
                  for f in range(1, N_FOLDS + 1)]
        summary.loc[name] = [roc_auc_score(y, pred), np.mean(scores),
                             np.std(scores, ddof=1)]
        fold_scores = pd.concat([fold_scores, pd.DataFrame({
            "fold": range(1, N_FOLDS + 1), "model": name, "auc": scores
        })], ignore_index=True)

print(summary.sort_values("auc_oof", ascending=False).round(6).to_string())
paired = fold_scores.pivot(index="fold", columns="model", values="auc")
if "hybrid" in paired:
    gains = paired[EXPERIMENTS].subtract(paired["hybrid"], axis=0)
    print("\nIncremento por bloque respecto de hybrid:")
    print(gains.round(6).to_string())
if "previous_xgb" in paired:
    gains_xgb = paired[names].subtract(paired["previous_xgb"], axis=0)
    print("\nIncremento respecto del XGBoost anterior:")
    print(gains_xgb.round(6).to_string())
    print("\nFolds ganados:", (gains_xgb > 0).sum().to_dict())

summary.to_csv(OUT / "metrics.csv")
fold_scores.to_csv(OUT / "fold_metrics.csv", index=False)


                  auc_oof  auc_fold_mean  auc_fold_std
model                                                 
hybrid_all       0.945159       0.945162      0.000708
hybrid_residual  0.944561       0.944564      0.000658
hybrid_income    0.943524       0.943528      0.000745
previous_optuna  0.942566       0.942583      0.000826
hybrid           0.941985       0.941990      0.000855
hybrid_profile   0.941946       0.941951      0.000862
hybrid_mobility  0.941943       0.941947      0.000864
previous_xgb     0.941852       0.941860      0.000826
additive         0.938854       0.938856      0.000904
previous_logit   0.938547       0.938549      0.000931

Incremento por bloque respecto de hybrid:
model  hybrid  hybrid_income  hybrid_mobility  hybrid_profile  hybrid_residual  hybrid_all
fold                                                                                      
1         0.0       0.001672        -0.000056       -0.000018         0.002811    0.003371
2         0.0       0.00

## 6. Diagnóstico por segmento y complemento al modelo anterior

El AUC dentro de cada segmento ayuda a localizar errores, pero no suma el AUC global: también importan los pares entre segmentos. Segmentos de una sola clase se reportan con AUC indefinido.

Un promedio fijo 90% XGBoost anterior + 10% candidato sirve como diagnóstico de complementariedad. No se buscan pesos sobre el mismo OOF ni se genera automáticamente una submission de ese promedio.

In [8]:
assert SUBMISSION_MODEL in names
candidate = oof[SUBMISSION_MODEL]
segment_rows = []
for col in ["Range_Anxiety_Level", "Subsidy_Available", "Home_Charging_Possible"]:
    for value in X[col].unique():
        mask = X[col].eq(value).to_numpy()
        enough = np.unique(y[mask]).size == 2
        row = {"variable": col, "value": value, "n": int(mask.sum()),
               "positives": int(y[mask].sum()),
               "auc_candidate": roc_auc_score(y[mask], candidate[mask]) if enough else np.nan}
        if "previous_xgb" in oof:
            row["auc_previous_xgb"] = (
                roc_auc_score(y[mask], oof["previous_xgb"][mask]) if enough else np.nan)
        segment_rows.append(row)
segments = pd.DataFrame(segment_rows)
print(segments.round(6).to_string(index=False))
segments.to_csv(OUT / "segments.csv", index=False)

if "previous_xgb" in oof:
    old = oof["previous_xgb"]
    mix = 0.9 * old + 0.1 * candidate
    print("\nAUC mezcla fija 90/10:", round(roc_auc_score(y, mix), 6))
    print("Delta sobre XGBoost:", round(roc_auc_score(y, mix)-roc_auc_score(y, old), 6))
    print("Spearman:", round(pd.Series(old).corr(pd.Series(candidate), method="spearman"), 6))
    print("Diagnóstico exploratorio; no prueba independiente tras seleccionar variantes.")


              variable  value      n  positives  auc_candidate  auc_previous_xgb
   Range_Anxiety_Level    Low 603972     114167       0.942576          0.939100
   Range_Anxiety_Level Medium  62499       2609       0.936674          0.932346
   Range_Anxiety_Level   High   2194          3       0.979918          0.980526
     Subsidy_Available     No 248756       1432       0.886040          0.874276
     Subsidy_Available    Yes 419909     115347       0.909639          0.904135
Home_Charging_Possible    Yes 462677      90601       0.941892          0.938394
Home_Charging_Possible     No 205988      26178       0.950238          0.947163

AUC mezcla fija 90/10: 0.942527
Delta sobre XGBoost: 0.000675
Spearman: 0.990219
Diagnóstico exploratorio; no prueba independiente tras seleccionar variantes.


## 7. Artefactos y submission

Se guardan predicciones de todas las variantes, IDs, folds, configuración, versiones y huellas de los archivos de entrada. La submission utiliza el candidato fijado al comienzo y el promedio de folds.

**El modo SMOKE no escribe una submission.** En modo completo, generar el CSV no implica que haya mejorado: revisar primero las métricas. No hay instalación de paquetes, llamadas a la API ni envíos automáticos.

In [9]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

payload = {"train_ids": train["id"].to_numpy(), "test_ids": test["id"].to_numpy(),
           "fold_ids": fold_ids, "y": y, "original_rows": original_rows}
for name in names:
    payload[f"{name}_oof"] = oof[name]
    payload[f"{name}_test"] = test_pred[name]
np.savez_compressed(OUT / "predictions.npz", **payload)

manifest = {
    "smoke": SMOKE, "seed": SEED, "outer_folds": N_FOLDS,
    "inner_folds": INNER_FOLDS, "alpha": ALPHA,
    "experiments": EXPERIMENTS, "submission_model": SUBMISSION_MODEL,
    "xgb_params": PARAMS, "versions": VERSIONS, "python": platform.python_version(),
    "additive": {"n_knots": 6, "degree": 3, "C": 0.1},
    "feature_blocks": BLOCKS,
    "input_sha256": {name: sha256_file(ROOT / "data" / name)
                     for name in ["train.csv", "test.csv", "sample_submission.csv"]},
}
(OUT / "config.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

if SMOKE:
    print("SMOKE terminado. Artefactos de prueba en:", OUT)
    print("No se generó submission. Ejecutar de nuevo con SMOKE=False para CV completa.")
else:
    submission = pd.DataFrame({
        "id": test["id"], TARGET: test_pred[SUBMISSION_MODEL]
    })
    assert list(submission.columns) == list(sample.columns)
    assert submission["id"].equals(sample["id"])
    assert len(submission) == len(sample)
    assert submission[TARGET].notna().all()
    assert submission[TARGET].between(0, 1).all()
    destination = ROOT / "submissions" / "5_hybrid_submission.csv"
    destination.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(destination, index=False)
    # Convención de los notebooks anteriores para futuros ensambles.
    np.save(ROOT / "models/5_hybrid_oof.npy", oof[SUBMISSION_MODEL])
    np.save(ROOT / "models/5_hybrid_test.npy", test_pred[SUBMISSION_MODEL])
    print("Submission:", destination)
    print("AUC OOF candidato:", roc_auc_score(y, candidate))
    if "previous_xgb" in oof:
        delta = roc_auc_score(y, candidate) - roc_auc_score(y, oof["previous_xgb"])
        print(f"Delta vs XGBoost anterior: {delta:+.6f}")
        print("Revisar estabilidad por fold antes de decidir un envío.")


Submission: c:\Users\HP\OneDrive\Escritorio\David Guzzi\Github\DGKaggle\Playground Series\Season 6\Episode 9 - Predicting Electric Vehicle Purchases\submissions\5_hybrid_submission.csv
AUC OOF candidato: 0.9451585586665369
Delta vs XGBoost anterior: +0.003307
Revisar estabilidad por fold antes de decidir un envío.


## 8. Resultados

- **`hybrid_all`: OOF 0,94516 (media por fold 0,94516 ± 0,00071) y score público 0,94544.** Es el mejor
  modelo del episodio. El leaderboard quedó 0,00028 **por encima** del OOF, así que la CV no sólo no
  se sobreajustó: subestimó.
- **+0,00259 sobre el XGBoost tuneado** (0,94257 OOF, 0,94242 público), y **+0,00302 medido en el
  leaderboard**. Gana en 5 de 5 folds con deltas de 0,00279 / 0,00265 / 0,00255 / 0,00235 / 0,00256 —
  desvío de la diferencia pareada 0,00016, un orden de magnitud menor que el desvío entre folds. La
  mejora es cuatro veces la que había dado el tuneo de hiperparámetros (+0,00077 en el leaderboard).
- **Toda la ganancia viene del ingreso.** Las ablaciones contra `hybrid` son inequívocas:
  `residual` +0,00257, `income` +0,00154, `mobility` −0,00004, `profile` −0,00004. Movilidad y perfil
  no aportan nada, y los dos bloques de ingreso son casi aditivos entre sí (+0,00317 juntos).
- **Los bloques que fallaron ya habían fallado antes.** El cruce `home_city` y el perfil de adopción
  (concern × subsidio × ansiedad) son exactamente los que el EDA había marcado como aditivos en
  log-odds y que el logit ya había medido en 0,0000. Tercera medición independiente, mismo resultado:
  esos cruces no existen como señal.
- **Hay estructura real a nivel de valor exacto de ingreso.** Agrupando por valor y residualizando
  contra una función suave, la varianza de las medias por grupo es **5,1 veces** la esperada si el
  ingreso actuara sólo de forma suave. No son registros duplicados: los pares que comparten ingreso
  coinciden en 4,16 de las otras 12 columnas contra 3,96 al azar — prácticamente nada. El efecto está
  pegado al valor en sí, que es lo que hace que una codificación por valor lo capture y las
  transformaciones suaves (splines, logaritmo, deciles) lo pierdan.
- **Y transfiere**, porque el **99,42 %** de las filas de test tienen un ingreso que ya aparece en
  train. Es lo que explica que el salto de CV se haya reproducido entero en el leaderboard.
- **El aditivo solo rinde 0,93885**, apenas +0,0003 sobre el logit del notebook 2. Los splines
  encontraron muy poca curvatura que el logit no tuviera ya: la relación era efectivamente casi lineal
  en log-odds, como había anticipado el EDA.
- **`hybrid` sin bloques da 0,94199, empatando con el XGBoost crudo (0,94185).** Es el control que
  sostiene todo el argumento: la arquitectura de dos etapas por sí sola no aporta nada, así que la
  ganancia es atribuible a las features y no al andamiaje.
- **La mejora se concentra donde el modelo viejo era más débil**: `Subsidy_Available = No` (248.756
  filas con sólo 1.432 positivos) pasa de 0,87428 a 0,88604, +0,0118 — el mayor salto por segmento.
  `Range_Anxiety_Level = Medium` gana +0,0043.
- **Distancia al tope**: el primer puesto está en 0,94672. La brecha pasó de 0,00430 a **0,00128**.

### Reservas

- **Sesgo de selección**: se eligió entre seis variantes mirando la misma CV. `SUBMISSION_MODEL` estaba
  fijado de antemano y `hybrid_all` ganó por 0,0006 sobre el segundo con 5/5 folds, así que el riesgo
  es bajo — pero el número exacto sigue teniendo algo de optimismo. El leaderboard, que es partición
  independiente, lo confirmó.
- **Asimetría train/test en las features aprendidas**: las filas de entrenamiento reciben la
  codificación de **un** mapa interno, mientras que validación y test reciben el **promedio de tres**.
  La versión promediada tiene menos ruido, así que el modelo se entrena sobre una feature más ruidosa
  de la que después ve. No es fuga y tiende a ser conservador, pero distorsiona cuánto se apoya
  XGBoost en ella.
- **`income_freq_exact` e `income_freq_100` están dentro de `INCOME_COLS`**, que la sección 2 describe
  como variables determinísticas. No lo son: se aprenden del split de ajuste. No dependen del target,
  así que no hay problema de fuga, pero la etiqueta engaña.
- **Los hiperparámetros no son los del notebook 4** (`max_depth=4` y 600 árboles fijos contra
  `max_depth=11` con early stopping). El control contra `hybrid` salva la atribución, pero queda sin
  medir cuánto más daría `hybrid_all` con los hiperparámetros tuneados.

El notebook usa exclusivamente los CSV locales de la competencia. No utiliza una fórmula externa supuestamente generadora del target.
